# Genomics on the command line day 2: alignment files
A quick reference sheet for all commands we will cover today is available [here](https://github.com/harvardinformatics/biotips/blob/main/genomeCL_day2_cheatsheet.md)

## Setting up the workshop
This workshop is organized as a `jupyter` notebook, which allows us to work in an interactive environment to edit and run code in small "blocks", without requiring you to install various packages yourselves. We'll be running it using [Google Colab](https://colab.research.google.com/): either click on the "Open Jupyter Notebook" link on the [landing page for the workshop](https://informatics.fas.harvard.edu/workshops/biotips/), or if you have downloaded the notebook yourself, just upload it to Colab and open it.

Jupyter notebooks have blocks of formatted text ("text" or "markdown" blocks), as well as "code" blocks that contain executable commands that can be run within the notebook. Clicking on the arrow in the upper left of a code block will execute the code in that block.

### Install the software
As before, we need to download a few tools to enable us to run the tools we will be discussion within this Colab notebook. Run the following code blocks by clicking the arrow in the upper left of each block.

In [ ]:
%%bash
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%bash
import condacolab
condacolab.check()

!conda install -c bioconda samtools

### Download the example data
Run the following code block to download the example datasets we will be using:

In [ ]:
%%bash
mkdir -p data_day2
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.bam
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.sam
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/reads_vs_reference.sam
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/large_alignment.bam

mkdir -p img
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/coverage_low_chromosome1.svg
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/coverage_zero_chromosome1.svg
wget -P img/ https://github.com/harvardinformatics/biotips/blob/80f12d94099041c5bbfe9cd2f73166d0c0f2b2bf/img/pairedReads.png
wget -P img/ https://github.com/harvardinformatics/biotips/blob/80f12d94099041c5bbfe9cd2f73166d0c0f2b2bf/img/read_coverage_cartoon.png

## Recap from day 1
Last time, we discussed:

- How files are stored in a Unix-like file system, 
- How to navigate a file system at the command line (`pwd`, `ls`, `cd` etc.)
- How input and output works, plus ways to visualize contents of files (`cat`, `less`, `wc -l`)
- The two most common bioinformatic sequence file formats, FASTA and FASTQ
- Common command line tools to manipulate them (`grep`, `sed`, `seqkit` etc.)

I've put together a [reference sheet](https://github.com/harvardinformatics/biotips/blob/main/genomeCL_day1_cheatsheet.md) of all the commands we covered in day 1 if you need to quickly refresh your memory!

### FASTQ files
I wanted to recap more in-depth on the final topic we covered last class, FASTQ files, as we only touched on them briefly and after quite a lot of other material! 

FASTQ is similar to FASTA format but also contains **q**uality information about the sequence. You will commonly encounter FASTQ files when working with sequencing pipelines, as second generation technologies like Illumina as well as 3rd generation technologies like PacBio and Nanopore output their data in this format. 

Each entry is comprised of *four* lines, as opposed to the two of FASTA:

-   *Header line* which starts with an `@` symbol and contains the sequence ID
-   *Sequence line* comprised of nucleotides
-   *Spacer line* which is just a `+` character (an optionally the sequence ID again)
-   *Quality line* which is a string of [ASCII characters](https://www.omixon.com/wp-content/uploads/2013/06/illumina_fastq_coding.png), each character corresponding to a base in the nucleotide sequence 

In [ ]:
%%bash 
cat data_day2/example.fastq

We only covered FASTQ files briefly, as most often you will not be doing too much manipulation of FASTQ files using generic command line tools, and specialized tools like `seqkit` handle FASTQ files almost identically to FASTA files. However, FASTQ files are very commonly the *starting point* of a bioinformatic pipeline: in many types of analysis you will have a FASTQ file that you want to map to another file. That brings us to today's topic!

## Alignment files
In this workshop, we will continue on in our "workflow" to discuss some common types of **alignment files**, which can be generated in a variety of ways, and like sequence files can represent a variety of things: whole genome alignments to identify rearrangements, RNAseq reads vs reference transcriptome to calculate differential expression, genomic resequencing mapped to a genome to identify polymorphism, even certain sequencers will output reads in an alignment format. 

Commonly, alignment files have a "reference" (sometimes also called a "target") sequence, with "query" sequence(s) mapped against the reference. In this case, mapping coordinates (i.e. where the align to) are relative to the reference sequence. For example, in an RNAseq experiment the reads in the individual RNAseq samples would be the **query** sequences that are mapped against a **reference** transcriptome.

In terms of file format, alignment files come in many types. A variant of FASTA files, which we discussed in the previous workshop, can even be used for alignments. In FASTA alignments, each sequence is aligned against every other sequence, and all sequences are made the same length by adding gaps (either `N` or `-` characters, frequently). Thus the sequence in an alignment FASTA file represents the consensus between all the sequences in the alignment.

This type of alignment file is useful for certain basic tasks (e.g. aligning gene sequences from different species to make a phylogenetic tree), but other file types are more specialized to represent alignments and are more information dense. Let's look at...


## Intro to SAM/BAM format
SAM (Sequence Alignment/Map) format is one of the most common file formats produced by many different pieces of alignment software, both for long and short read sequence data. A number of different programs can output alignments in this format (e.g. [BWA](https://github.com/lh3/bwa), [STAR](https://github.com/alexdobin/STAR), [minimap2](https://github.com/lh3/minimap2)) and which you choose will vary based on your data type and experimental design, but the alignment file created will likely be interchangeable. In all cases, one sequence is designated as the reference, with queries aligned against it.

It is a tab delineated text file, with 11 mandatory fields, or columns, (listed below), with a 12th column containing optional "tags" with supplemental or aligner-specific information. SAM files are human readable, but can be quite large. An alternate format is the Binary Alignment/Map (BAM) file, which is binary compressed and not human readable, but is more compact and efficient to work with. Most pipelines will use BAM format over SAM, and for storing alignments long-term BAM is usually preferable as it uses up less storage space. Converting between BAM and SAM is easy, so there is no need to have both versions.

| **Column** | **Description**                        |
|------------|----------------------------------------|
| 1          | Read name                              |
| 2          | Bitwise flag                           |
| 3          | Reference name                         |
| 4          | Leftmost mapping position              |
| 5          | MAPQ quality score                     |
| 6          | CIGAR string                           |
| 7          | Name of 2nd read in pair               |
| 8          | Position of 2nd read in pair           |
| 9          | Length of mapping segment              |
| 10         | Sequence of segment                    |
| 11         | Phred33 quality score at each position |
| 12         | Optional tags                          |

In addition to these tab-separated fields, BAM/SAM files also have header lines at their starts, which are denoted by a `@` character and a two letter code. Header lines contain metadata about the alignment, such as whether alignments are sorted or grouped, reference sequence information, and definitions for any tags that are used in the 12th column. While not strictly required, many downstream analysis tools require BAM/SAM files to have a header, and will thrown an error if not present. A full description of BAM/SAM format can be found [here](https://samtools.github.io/hts-specs/SAMv1.pdf).

Many different types of bioinformatic analysis will use BAM/SAM alignments, such as:
- Identifying structural variation or SNPs in populations
- Calculating differential gene expression
- Verifying genome assemblies
- Annotating genomes...

Let's take a look at what these files look like.

> Use `less` to open SAM file (using the `-S` argument to wrap lines). Do the same for BAM file.

In [ ]:
%%bash
less -S data_day2/example.sam

In [ ]:
%%bash
less -S data_day2/example.bam

Just like any file, we can use our normal command line tools to visualize SAM files...however, it is a little clunky. Also, notice that when we try to open the BAM file, we get a scrambled mess; this is because as mentioned, BAM is a binary compressed format and is not human readable (which the shell will warn you of when trying to open the file). Thus, when working with BAM/SAM alignments, it is much easier to use a toolkit specilized for these types of files!

## SAMtools
[SAMtools](http://www.htslib.org/doc/samtools.html) is a suite of programs that are extremely useful for processing mapped reads and for downstream analysis. As stated above, BAM/SAM files from different programs are (mostly) interchangeable, so `samtools` will work with a file BAM/SAM file no matter what program produced it. Note that for this workshop we have already installed `samtools`, but to run it elsewhere you will need to install it yourself. It has a ton of functions (which you can check out on the [manual page](http://www.htslib.org/doc/samtools.html)), but we will go through several of the most common uses.

### samtools view
As the name suggests, this command lets you view the content of a SAM **or** BAM file. The most basic syntax for the command looks like this:

```
samtools view [ARGUMENTS] [INPUT_FILE.BAM/SAM]
```

When run on a BAM/SAM file, `view` will output the contents of the file to the screen (i.e. **standard output**). As these files can get quite large we often don't want to print the entire thing, so let's take a look at a SAM file by opening it with `view` and **piping** it to `head` to display just the first five lines:

In [ ]:
%%bash
samtools view data_day2/example.sam | head -n 5

We can see the first five alignments in the SAM file, where each line represents a sequence (in this case, a sequencing read) aligned against a reference: we can information about each alignment such as the aligned read name, coordinates in the reference it aligns to, info about how well it aligns, etc.

These are the alignment lines, but as mentioned BAM/SAM files also have *header lines* that we will want to see as well, which by default `samtools view` does NOT display. Let's add the `-h` argument to our `view` command, which will display the header as well:

In [ ]:
%%bash
samtools view -h data_day2/example.sam | head -n 5

As we saw when trying to open the files with `less`, SAM files are already human-readable, so technically we don't need to use `samtools view`...however, we can also use it for the binary-compressed BAM files which are NOT human-readable:

In [ ]:
%%bash
samtools view data_day2/example.bam | head -n 5

Note that even tho we are opening a BAM file, `view` automatically converts it into readable SAM format; in other words, by default `samtools view` converts to BAM --> SAM! So if we wanted to convert a BAM file to SAM and save it as a new file, we can use `samtools view` with the `-o` argument plus the desired name for our new file:

In [ ]:
%%bash
#This will create a new file, converted.sam
samtools view -o converted.sam data_day2/example.bam


To convert the other direction from SAM --> BAM, we can use the `-b` argument to instead output in BAM format, along with the `-o` argument:

In [ ]:
%%bash
samtools view -b -o converted.bam data_day2/example.sam

However, we are missing one important detail in creating these new `converted.bam/sam` files...can you spot what it is?

<details><summary>Solution</summary>

The output files do not include header lines! As mentioned, headers are vital for many downstream tools, and if we create any other BAM/SAM files from a file lacking a header they will also lack headers, so always be vigilant with your `view` arguments. To make a proper BAM file from a SAM file, we would instead want:
  
</details>

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
samtools view -h -b -o converted.bam data_day2/example.sam

#### Building intuition: what's wrong with my BAM file?
Just like we did in the previous workshop, we want to try to work on cultivating that bioinformatic "sixth sense" that helps us identify when something doesn't look right and where/how to troubleshoot. Doing this with BAM/SAM files is tricky, as the format is more complex and there are many areas that problems can arise. Broadly, I would categorize errors in BAM files as falling into two categories:

- **Format errors**: something is wrong your `samtools` syntax or with the file itself, e.g.:
  - Truncated or malformatted files
  - Missing header or info in the header
  - Confusing SAM and BAM
- **Alignment errors**: something is wrong with your data, e.g.:
  - Reference or queries are incorrect
  - A mistake when generating the alignment (e.g. used wrong settings)
  - Your data is bad

Periodically throughout this workshop, we'll walk through some hypothetical situations you might find yourselves in when working with these kinds of files and how to diagnose and the fix the problems!


In this scenario, imagine you have a SAM file of RNAseq data aligned to a reference transcriptome that you are trying to use to calculate differential expression between different tissues. You convert the SAM file to BAM format and feed the BAM file into a software pipeline that will calculate expression...

```
samtools view -H -o data_day2/sample.bam data_day2/example.sam
```

However when you run your differential expression pipeline, the program gives the message `ERROR: the specified BAM file is empty!`

**Discussion**: what is a logical place to start troubleshooting the issue?

<details><summary>Solution</summary>

As always, the best thing to start with is *looking at our files*, so let's check both our input and output:

```
samtools view data_day2/example.sam | less

samtools view data_day2/sample.bam | less
```

We can see that while our source file does contain alignments, the BAM file that we created from it is indeed empty. My next guess when troubleshooting would be that Something is wrong with the formatting somewhere, so I would want to check the headers as well to see if something jumps out at me:

```
samtools view -h data_day2/example.sam | less

samtools view -h data_day2/sample.bam | less
```

Doing this, we can see that our BAM file isn't TOTALLY empty, it actually does have header lines, but no alignments! This indicates to me that I have probably done something silly with the actual `view` command, and sure enough if we look at it:

```
samtools view -H -o data_day2/sample.bam data_day2/example.sam
```

We can see that I made a small typo, using upper case `-H` instead of the proper lower case `-h` when adding the header, and if we look at the manual we can see that the `-H` option ONLY prints the header!

  
</details>

#### Quick review: paired FASTQ files
At the end of the previous workshop we briefly touched on a special type of sequence files, *paired FASTQ files*. These are most commonly generated from Illumina short read sequencing, where each sequencing read has a partner or "mate" read associated with it that is a known distance away:

![Paired-end short reads](img/pairedReads.png)
[Source](https://www.illumina.com/science/technology/next-generation-sequencing/plan-experiments/paired-end-vs-single-read.html)

When aligning paired-end data, we can use information about how each read in a pair maps: do both reads map? In the correct orientation and right distance apart? This information becomes relevant when discussing how we can manipulate alignment files, such as...

### Parsing alignments using flags
BAM/SAM files can be a little overwhelming to look at as they contain tons of information about each alignment, only some of which will be relevant or useful. One of the most important columns that you will want to pay attention to is the second column in a BAM/SAM file, which is called the *bitwise flag*. The flag value is an integer, which is the *sum of a series of integer values that give information about how a read is mapped*.

| **Integer** | **Description**                |
|-------------|--------------------------------|
| 1           | read is paired                 |
| 2           | read mapped in proper pair     |
| 4           | read unmapped                  |
| 8           | mate is unmapped               |
| 16          | read on reverse strand         |
| 32          | mate on reverse strand         |
| 64          | first read in pair             |
| 128         | second read in pair            |
| 256         | not primary alignment          |
| 512         | alignment fails quality checks |
| 1024        | PCR or optical duplicate       |
| 2048        | supplementary alignment        |

For each alignment in the file, it will have the associated integer values for every criteria in the above table that the alignment fulfills. So e.g., for a paired-end mapping data set, if a paired read (**1**) is mapped in proper pair with its mate (**2**), and it is the first in the pair (**64**) and its mate properly maps on the reverse strand (**32**), it will have a value = **99** (1+2+32+64) in the second column. Don't worry about memorizing these, there are plenty of tools online that decode these flags for you, such as right [here](https://broadinstitute.github.io/picard/explain-flags.html).

While you don't need to know all the SAM flags, if there is one flag that is useful to have memorized it is **4**, which means the read is **unmapped**. Unmapped reads are most often filtered out, as many programs used in downstream analysis of SAM/BAM files only want mapped reads (and also to save space on disk!). 

We can use two arguments to `samtools view` to filter our alignments based on a specified flag:
- `-f [FLAG]`: only print alignments that contain `[FLAG]`
- `-F [FLAG]`: only print alignments that do NOT contain `[FLAG]`

For example, the following command will print reads that do not contain the flag **4**, i.e. it *only prints aligned reads*: 

In [ ]:
%%bash
samtools view -F 4 data_day2/example.sam | head

Note that filtering on a particular flag works not just for that literal flag, but for *any integer sum that contains that flag*. For example, `-F 4` will filter out any alignment with the `4` flag, but also e.g.: 

- `77` (1+**4**+8+64) = "read paired + read unmapped + mate unmapped + first in pair" 
- `141` (1+**4**+8+128) = "read paired + read unmapped + mate unmapped + second in pair"
- `101` (1+**4**+32+64) = "read paired + read unmapped + mate on reverse strand + first in pair"

Flags can be a little tricky to wrap your head around, so let's do some exercises that focus on the practical questions you can answer about your alignment files by looking at their flags!

> **Exercise**:
> Write a command that counts the number of unmapped reads in the file `data_day2/example.bam`. Hint: you will need a `samtools view` command, plus a **pipe** to a command we covered last week!

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
samtools view -f 4 data_day2/example.bam | wc -l

> **Exercise**:
> In the code block below, count how many reads in the SAM file `data_day2/example.bam` are mapped in their proper pairs vs not proper pairs? (Hint: this will require two commands...)

In [ ]:
%%bash
# command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
samtools view -f 2 data_day2/example.bam | wc -l
samtools view -F 2 data_day2/example.bam | wc -l

> **Exercise**:
> In the code block below, write a command to count how many reads are mapped but their mates are NOT mapped (i.e. how many **singleton** reads are there) in the file `data_day2/example.bam`:

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
#-f 8 prints reads where the mate is unmapped
#-F 4 excludes reads where the read itself is unmapped
samtools view -f 8 -F 4 data_day2/example.sam | wc -l

### Developing intuition: what's wrong with my BAM file?
Imagine you are studying nucelotide diversity within a population. A collaborator has generated a BAM alignment file by mapping whole genome sequencing data from 100 individuals in the population against a reference genome, which you want to use for analysis. You download the BAM file from your collaborator's lab server to your local machine and take a look at it:

```
ls -lh data_day2/large_alignment.bam
```

**Discussion**: what do you notice that is suspicious?

<details><summary>Solution</summary>

Given that the BAM is supposed to be an alignment of a hundred samples, we would expect the file to be quite large, probably 100s of GB in size, and the file the we downloaded is *way too small*. When downloading files from a server (especially large files), it is not uncommon for a download to get interrupted and the file to be truncated. You will probably need to re-download the file.

To check your file, BAM/SAM files contain a 28-byte `E`nd `O`f `F`ile (`EOF`) marker at the end of the file to mark that the file is complete. `samtools` has a built-in tool that will verify the `EOF` is present:

```
samtools quickcheck data_day2/large_alignment.bam
```

Also, when downloading a file from a server, frequently the file uploader will include a `checksum`. A `checksum` is essentially a unique "fingerprint" that is generated from a file using an algorithm that looks like a string of integers. If you know what algorithm was used to generate the checksum (e.g. `md5`, one of the most common algorithms), you can generate a fingerprint of your copy of the file and compare it to the uploader's to verify your file's integrity...if they do not match, it means your file was altered.
  
</details>

### Sorting and indexing a BAM file
BAM files can get extremely large, over hundreds of GB in some cases, and by default are unsorted, which makes any task where we need to search for specific regions too computationally intensive. To overcome this, we can use two other functions of `samtools`, `sort` and `index`. This will create a BAM index `.bai` file, which allows quick lookup even for very large BAM files.

In [ ]:
%%bash
# -o: This option tells samtools sort to print the output to the provided file rather than to the screen
samtools sort -o file.sorted.bam data_day2/example.bam

#samtools index does not have an -o option, and will automatically create an index file with the same name as the input BAM file with a .bai extension
samtools index file.sorted.bam

This code will create a new `.sorted.bam` file that we create the index for, as `samtools index` requires a coordinate-sorted file. Any downstream program that *refers to specific regions of interest* in a BAM file, such as visualization tools like IGV (discussed later) or other `samtools` functions will require this index. For example, we can specify *specific region(s)* when using `samtools view` to only print alignments that overlap the specified region. Regions are listed at the end of the `view` command with the format `reference name:start position-end position` (if start and end are not specified, it reports all alignments to the reference sequence): 

In [ ]:
%%bash
#Outputs all alignments that are mapped to chromosome 1 and saves them to a new BAM file
samtools view -o chr1.bam data_day2/example.bam chr1


In [ ]:
%%bash
#Outputs all alignments that are mapped from bases 1000 to 2000 (inclusive) on chromosome 1 and saves them to a new BAM file
samtools view -o chr1_subregion.bam data_day2/example.bam chr1:1000-2000

> PRACTICE: Putting it all together!

Let's take everything that we have learned and organize it into what a typical workflow might look like. Assume that we have already aligned the data and the aligner we used outputs the alignment file in SAM format. We want to go from our initial SAM file and end up with a *sorted, indexed BAM file with only the mapped reads retained*. Try inputting the commands yourself, then we will walk through it together.

> **Exercise**:
> 1. Convert the `data_day2/example.sam` **SAM** file to **BAM** format while retaining the header and removing unmapped reads
> 2. Then sort the file and save it as a final file called `file.mapped.sorted.bam`.
> 3. Index the newly created sorted **BAM** file.
> (Optional bonus: `samtools` commands can also make use of **pipes** (`|`), so try doing the filtering and sorting in a single line to avoid writing intermediate files!)

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}

## Convert to BAM with header and without unmapped reads and then sort
samtools view -h -b -o file.mapped.bam -F 4 data_day2/example.sam
samtools sort -o file.mapped.sorted.bam file.mapped.bam

#Or, in a single line:
samtools view -h -b -F 4 data_day2/example.sam | samtools sort -o file.mapped.sorted.bam 

## Index the new BAM file
samtools index file.mapped.sorted.bam

### More useful `samtools` functions
We have our nice sorted and indexed BAM file, now what are some other useful pieces of information we can pull out of it? 

#### `samtools stats`
As the name suggests, the `stats` function calculates some basic summary statistics about a BAM file, such as number of sequences aligned, number of reads mapped in proper pairs (if using paired-end data), average alignment error rate, average alignment quality, plus lots more which are all described in the [manual page](https://www.htslib.org/doc/samtools-stats.html). 

In [ ]:
%%bash
samtools stats data_day2/example.bam > data_day2/example.bam.stats

> **Exercise**
> We are only interested in the "summary numbers" section of the stats output file. Thinking back to last week's workshop, write a command that pulls the summary numbers out of `samtools stats` output (hint: remember one of the tools we covered in the last workshop and check the manual page or look thru the file to find the relevant pattern)

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
samtools stats data_day2/example.bam | grep '^SN' > data_day2/example.bam.stats.txt

#### Converting to sequence files
We know that BAM/SAM files contain sequence names, nucleotide sequence and corresponding quality scores...in other words, everything we need to make a FASTA/FASTQ file! Accordingly, we can use the built-in function `samtools fastq` and `samtools fasta`:

```
samtools fastq -o output.fastq data_day2/example.bam
```

This would output to a single outfile, such as when we have unpaired reads. For paired end reads, we would instead want:

```
samtools fastq -1 reads1.fastq -2 reads2.fastq data_day2/example.bam
```

> **Exercise**:
> Like any other `samtools` functions, we can use `fastq/a` in combination with other tools. Write a command that outputs only the mapped reads in a BAM file to a FASTQ file (BONUS: only output reads that are mapped, primary alignments and non-supplementary alignments). The reads are unpaired.

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
samtools view -F 4 data_day2/example.bam | samtools fastq -o mapped_reads.fastq

#Bonus: 4 + 256 + 2048 = 2308 (unmapped, not primary alignment, and supplementary alignment flags)
samtools view -F 2308 data_day2/example.bam | samtools fastq -o mapped_reads.fastq

### Calculating read coverage
In many kinds of bioinformatic analysis, we will be interested in the **read coverage** and **read depth** in a particular region. "Coverage" and "depth" are often used interchangably but are technically are different concepts. "Coverage" is defined as the percentage of positions that have *at least one base aligned to it* (think of it as how much sequence is covered by mapped reads), while "depth" can be thought of as the redundancy of coverage (i.e. how many bases are aligned to a particular sequence). 

![Depth vs coverage](img/read_coverage_cartoon.png)

#### Why do we care about coverage/depth?
Identifying areas of abnormally low coverage/depth or abnormally high coverage/depth can tell us lots of different information. Aligning vs a reference and calculating coverage and/or depth for an alignment is the starting point for mainy kinds of analysis, such as:

- Mapping genomic reads against an assembly to identify possible misassemblies
  - In misassembled regions, we'd expect no reads to align (i.e. zero coverage)
- Calculating differential expression of transcripts
  - Transcripts with higher depth == more highly expressed
- Calling single nucleotide polymorphisms or structural variants within a population
  - If variant has adequate depth, it can be distinguished from random error 


To calculate these stats, we can use (appropriately enough) `samtools coverage` and `samtools depth`! 

As we can see from the [manual page](https://www.htslib.org/doc/samtools-coverage.html), the `coverage` command calculates a summary of both coverage and depth within a specified region, or over the entirety of each reference sequence if no regions are provided. E.g.:

In [ ]:
%%bash
samtools coverage data_day2/example.bam

In [ ]:
%%bash
samtools coverage -r chr1:1000-2000 data_day2/example.bam

> Note that unlike with `samtools view` where we specify a region at the end of the command, `coverage` requires it as an argument with `-r`...this is a good reminder that unfortunately command syntax won't always be consistent even within the same software package, and to always check the manual!

We can see that this produces a table (technically a *tab separated value* [`.tsv`] list, which we will talk about in the next workshop!) that summarizes the coverage over each reference chromosome, or within the specific subregion if specified:

#rname startpos endpos   numreads   covbases   coverage   meandepth   meanbaseq   meanmapq
chr1   1        1000000   4853      280820     28.082     0.4853      40          40
chr2   1        1000000   4853      280820     28.082     0.4853      40          40
chr3   1        1000000   4852      280780     28.078     0.4852      40          40
chr4   1        1000000   2426      222980     22.298     0.2426      40          40

- `numreads`: number of reads aligned
- `covbases`: number of bases in ref sequence covered by at least 1 reads
- `coverage`: percentage of ref sequence covered by at least 1 read
- `meandepth`: average depth of coverage across whole ref sequence
- `meanbaseq`: average Q score (i.e. quality) of reference bases
- `meanmapq`: average Q score of reads

While a summary of coverage/depth is useful, often we will want to know to know depth at each *individual position* in the reference, as that will to identify specific loci where depth is abnormally high or low. For this we use `samtools depth`, which will return a 3 column list of reference sequence, numeric position (i.e. base 1, 2, 3, etc.) and the depth at that base:

In [ ]:
%%bash
#The -a argument tells samtools to output depth at all positions, including positions with zero depth
#-o outputs the list to a file instead of the terminal
samtools depth -o data_day2/example.bam.depth.txt -a data_day2/example.bam

head data_day2/example.bam.depth.txt

### Developing intuition: what's wrong with my BAM file?
Say that you are studying genomic structural variation (inversions, translocations, etc.) within a population. You have a good quality reference genome, as well as whole genome resequencing (non-paired end) for several different individuals of the same species. In this example:

- The reference genome (FASTA format) is 4 megabases in length
- Each resequencing individual has a FASTQ file containing ~240 megabases of sequence 

By aligning the resequencing data against the reference, you want to use read depth to characterize a suspected structurally complex region on the first chromosome. To test this approach, you select one individual and align the reads against the reference using the following pipeline:

In [ ]:
%%bash
#Alignment step (don't run this, for demonstration only)
minimap2 reference.fasta reads.fastq > reads_vs_reference.sam

#Convert to BAM, sort and index
samtools view -b -h -f 4 -o reads_vs_reference.mapped.bam reads_vs_reference.sam

samtools sort -o reads_vs_reference.mapped.sorted.bam reads_vs_reference.mapped.bam
samtools index reads_vs_reference.mapped.sorted.bam

#Calculate depth and subset chromosome 1
samtools depth -o reads_vs_reference.depth.txt -a reads_vs_reference.mapped.sorted.bam
grep -w 'chr1' reads_vs_reference.depth.txt > chr1.depth.txt

#Make plot of coverage (don't run this, for demonstration only)
Rscript make_coverage_plot.R chr1.depth.txt > chr1.depth.png

The pipeline runs without errors, and the coverage plot looks like this:

![Zero coverage](img/coverage_zero_chromosome1.svg)

**Discussion**: what is the problem here? What are some possible things that could be going wrong? Where should we start in troubleshooting?

<details><summary>Solution</summary>

We are seeing zero coverage across the chromosome, which definitely is not right!

The actual problem could be from several different sources:
- The alignment step was incorrect
- The BAM file is corrupted/empty
- The visualization script is doing something wrong
- Something is wrong with the sequencing data itself

Here are the steps I would take when troubleshooting:
- Inspect the files using `samtools view`, make sure there is stuff in it
- Get some summary info about alignment files at each step using `samtools stats`:

```
samtools stats reads_vs_reference.mapped.sorted.bam | grep '^SN'

samtools stats reads_vs_reference.mapped.bam | grep '^SN'

samtools stats reads_vs_reference.sam | grep '^SN'
```
  
- Check the commands run:
  - Is the alignment step using the correct data and reference?
  - Was there some filtering done at any step?
  - Does my plotting script work properly?
  - Is there something wrong with the data itself?
  
</details>

<details><summary>Solution</summary>

Looking at our pipeline, let's say that the alignment step is correct and the plotting script should work correctly (as that is outside the scope of this workshop!).

Checking our BAM file, we can see that there are entries in it...

```
samtools view reads_vs_reference.mapped.sorted.bam | less
```

However! Looking closer, we can see that the reads in the BAM file *are all unmapped*. In addition, running `samtools stats` on each alignment file shows that we are losing reads when we go from SAM to BAM (i.e. the numbers do not add up).

And when we look at the `view` command in our pipeline, we can see that the source of the error is due to a mistake in filtering: we were trying to remove unmapped reads by filtering on the `4` flag, but we used the *wrong argument*.

We wanted `-F 4` (which will exclude alignments with the `4` flag) but instead we used `-f 4` (which only includes alignments with the `4` flag)!

This is a classic mixup when working with BAM files, and shows how a subtle syntax error can cause a problem that is only noticed at a later point.
  
</details>

**Discussion**: let's consider a related scenario. You are running the same pipeline as above (with the errors fixed!) and generate a plot of depth of coverage over chromosome 1, as before. This time, instead of zero coverage, the plot looks like this:

![Low coverage](img/coverage_low_chromosome1.svg)

Can we spot any potential problems with this data now?

<details><summary>Solution</summary>

The first thing we should ask ourselves is *what do we expect coverage to look like*? In this scenario we know our reference genome is 4 Mbase and the sample we are aligning contains about 240 Mbase. We can get an expectation for average depth of coverage by:

240 Mbase / 4 Mbase = **60X coverage**

Unlike before, we do see coverage across the chromosome in our plot...however, based on the amount of data in our FASTQ file, we expect average depth of coverage to be ~60X, so the average we are seeing seems *too low*.
  
</details>

**Discussion**: what are some things we think could be going wrong? How does troubleshooting this problem compare to the initial example? What should we be looking for in this case and what are some steps we could take?

<details><summary>Solution</summary>

I would say that while the "things that could have gone wrong" are similar between the two examples, having unexpectedly low coverage is a more complex problem, as having zero coverage is likely to stem from a more straight-forward mistake (such as the typo in our above example).

Troubleshooting should look similar to the first example:
- Check alignment stats at each step with `samtools stats`
  - We can see that in this case, the numbers "add up" properly and we aren't accidentally discarding reads we shouldn't be...
  - In other words, the problem does not lie with the alignment file

The exact solution is beyond the scope of this workshop, but after ruling out a problem with the BAM/SAM files I would start checking my reference and the reads themselves:
- Run a QC program on the FASTQ read files: is the data low quality?
- Is my reference genome correct? Did I accidentally use the wrong one?
  - Same for the reads
- Is the aligner software working properly? Did I use the wrong parameters?
  
</details>

In the above examples, we are using the external program `R` to visualize coverage over our reference from the file created by `samtools depth`, which we could also accomplish using a `python` script or some other language like `matlab`. While the detail can vary based on what kind of analysis you are doing, this is a common approach when plotting read depth or coverage, as making plots at the command line is very clunky. However, let's explore one other method we can use to visualize coverage.

### Visualizing alignments with IGV
Although it isn't technically a command line program and is not part of `samtools` or even limited to just alignment files, I wanted to introduce the `I`ntegrative `G`enome `V`iewer (`IGV`) program, as it is probably the most widely used graphical user interface to visualize BAM files. Also, while writing your own scripts for vizualization in `R` or `python` is useful, sometimes have an interactive interface to explore alignments and "eyeball" your what they actually look like is essential. `IGV` is its own standalone program (which you can download [here](https://igv.org/doc/desktop/), though it also requires `java` to be installed) and has way too much functionality to go over fully, but we will go over the basics of visualizing a BAM alignment.

Once we have installed and opened `IGV`, we will need three files: the reference genome in FASTA format, the BAM alignment we want to vizualize and the corresponding index file (which should have the same named prefix as the BAM plus the `.bai` extension and be in the same directory).

- First click the `Genomes` tab --> `Load Genome from File` --> navigate to and select the genome FASTA file
- Click `File` --> `Load from File` --> navigate to and select the BAM file

We should see a track with representations of reads mapped to the reference and a histogram showing depth of coverage. We can select different reference chromosomes or specific regions using the menu bars at the top center, use the mouse to pan along the sequence, zoom in and out, etc. We can also look at individual mapped reads, which have bases colored whether they match reference (by default, grey = same as reference) as well as any gaps or indels. 

Again, this is just a super quick overview of what `IGV` can do to get you aware that it is an option! 